In [14]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error
from scipy.linalg import svd, pinv
import pandas as pd

# ==========================================
# 1. SVD / POD: Dimensionality Reduction
# ==========================================
def perform_svd(X_data, variance_threshold=0.999):
    """
    X_data: shape (N_spatial_points, N_samples)
    Returns truncated basis U_r, singular values, and latent coordinates C_r
    """
    print("Performing SVD on wavefunction snapshots...")
    U, S, Vh = svd(X_data, full_matrices=False)
    
    # Calculate cumulative variance to find required rank r
    cumulative_variance = np.cumsum(S**2) / np.sum(S**2)
    r = np.argmax(cumulative_variance >= variance_threshold) + 1
    
    print(f"Truncation rank r={r} captures {variance_threshold*100}% of variance.")
    
    U_r = U[:, :r]             # POD Basis (Modes)
    C_r = np.diag(S[:r]) @ Vh[:r, :] # Latent coordinates for all samples
    
    return U_r, S, C_r, r

# ==========================================
# 2. Neural Network Surrogate (Data-Driven)
# ==========================================
def train_neural_network(params_train, C_r_train):
    """
    params_train: shape (N_samples, 3) -> [kappa, q, sigma]
    C_r_train: shape (N_samples, r) -> transposed to (N_samples, r) for sklearn
    """
    print("Training Neural Network (Parameters -> POD Coordinates)...")
    
    # Using a standard Multi-Layer Perceptron
    nn_model = MLPRegressor(
        hidden_layer_sizes=(64, 64), 
        activation='tanh', 
        max_iter=2000, 
        learning_rate_init=0.001,
        random_state=42
    )
    
    # Train model to predict the POD coordinates (C_r) from physical params
    nn_model.fit(params_train, C_r_train.T)
    
    # To predict a new wavefunction:
    # 1. c_pred = nn_model.predict(new_params)
    # 2. phi_pred = U_r @ c_pred.T
    return nn_model

# ==========================================
# 3. Reduced Basis Method (Physics-Informed)
# ==========================================
def rbm_galerkin_projection(U_r, T_full, V_trap_full):
    """
    U_r: Truncated POD basis from SVD
    T_full: High-dimensional Kinetic Energy Matrix (-d^2/dx^2)
    V_trap_full: High-dimensional Harmonic Trap Matrix (kappa * x^2)
    """
    print("Performing Galerkin Projection for RBM...")
    
    # Project linear operators into the r x r subspace
    T_r = U_r.T @ T_full @ U_r
    V_trap_r = U_r.T @ V_trap_full @ U_r
    
    # NOTE: As you correctly anticipated, the nonlinear term (q * |phi|^sigma) 
    # cannot be pre-computed easily. In a standard RBM without DEIM 
    # (Discrete Empirical Interpolation Method), you must reconstruct phi, 
    # evaluate the nonlinearity, and project it back at each iteration:
    # V_nl_r = U_r.T @ (q * np.abs(U_r @ c_k)**sigma * (U_r @ c_k))
    
    return T_r, V_trap_r

# ==========================================
# 4. Dynamic Mode Decomposition (DMD)
# ==========================================
def perform_dmd(X_solver_iterations, r_dmd=5):
    """
    Applies DMD to the intermediate steps of the SCF solver for ONE parameter set.
    X_solver_iterations: shape (N_spatial_points, N_iterations)
    """
    print("Performing DMD on SCF solver intermediate iterations...")
    
    # Create time-shifted matrices
    X1 = X_solver_iterations[:, :-1]
    X2 = X_solver_iterations[:, 1:]
    
    # 1. SVD on X1
    U, S, Vh = svd(X1, full_matrices=False)
    U_r = U[:, :r_dmd]
    S_r_inv = np.diag(1.0 / S[:r_dmd])
    V_r = Vh.T[:, :r_dmd]
    
    # 2. Compute reduced A matrix (Atilde)
    Atilde = U_r.T @ X2 @ V_r @ S_r_inv
    
    # 3. Eigen-decomposition of Atilde
    eigenvalues, W = np.linalg.eig(Atilde)
    
    # 4. Reconstruct DMD Modes (Phi)
    Phi = X2 @ V_r @ S_r_inv @ W
    
    # Eigenvalues map to the convergence/decay rates of the SCF solver
    convergence_rates = np.log(eigenvalues) 
    
    return Phi, eigenvalues, convergence_rates

# ==========================================
# Example Execution Pipeline
# ==========================================
if __name__ == "__main__":
    # --- MOCK DATA SETUP (Replace with your actual GPE data) ---
    N_space = 150
    N_samples = 125 # 5x5x5 grid
    
    # Mock wavefunctions and parameters
    X_data_raw = pd.read_csv('gpe_train_data.csv')['phi']
    parsed_rows = [np.fromstring(row.strip('[]'), sep=' ') for row in X_data_raw]

    # Stack the 1D arrays back into a proper 2D numpy matrix
    X_data = np.vstack(parsed_rows).T
    params = np.array([pd.read_csv('gpe_train_data.csv')['kappa'], pd.read_csv('gpe_train_data.csv')['q'], pd.read_csv('gpe_train_data.csv')['sigma']]).T # [kappa, q, sigma]
    
    # Mock PDE Operators for RBM
    T_full = np.eye(N_space) # Replace with actual 2nd derivative matrix
    V_trap_full = np.eye(N_space) # Replace with actual kappa*x^2 matrix
    
    # Mock SCF solver iterations for DMD (e.g., 50 iterations for a single state)
    X_scf_history = np.random.rand(N_space, 50) 
    
    # --- RUN PIPELINE ---
    # 1. SVD
    U_r, S, C_r, r = perform_svd(X_data, variance_threshold=0.999)
    
    # 2. Neural Network
    nn_surrogate = train_neural_network(params, C_r)
    
    # 3. RBM Projection
    T_r, V_trap_r = rbm_galerkin_projection(U_r, T_full, V_trap_full)
    
    # 4. DMD on Solver
    DMD_modes, DMD_eigs, decay_rates = perform_dmd(X_scf_history, r_dmd=3)
    
    print("\nPipeline execution complete. Ready for extrapolation testing.")

Performing SVD on wavefunction snapshots...
Truncation rank r=2 captures 99.9% of variance.
Training Neural Network (Parameters -> POD Coordinates)...
Performing Galerkin Projection for RBM...
Performing DMD on SCF solver intermediate iterations...

Pipeline execution complete. Ready for extrapolation testing.


In [18]:
# Mock wavefunctions and parameters
X_test_raw = pd.read_csv('gpe_interp_test_data.csv')['phi']
parsed_rows = [np.fromstring(row.strip('[]'), sep=' ') for row in X_data_raw]

# Stack the 1D arrays back into a proper 2D numpy matrix
X_test = np.vstack(parsed_rows).T

In [13]:
params.shape

(125, 3)